In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

dbutils.fs.ls("abfss://bronze@grtadlsdev.dfs.core.windows.net/")

df = spark.table("bronze.calendar_dates")
display(df.limit(50))

# null counts
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

In [0]:
df.printSchema()

In [0]:
df = df.withColumn("date", F.to_date(F.col("date").cast("string"), "yyyyMMdd"))

display(df.limit(50))
df.printSchema()

In [0]:
spark.sql("SHOW TABLES IN silver").show()

In [0]:
%sql
create schema if not exists silver;

In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema" ,"true").saveAsTable("silver.calendar_dates")

silver_path = "abfss://silver@grtadlsdev.dfs.core.windows.net/calendar_dates/"
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver_path)